# 🤝 Module 6: Multi-Agent Systems

Build an Orchestrator that routes tasks to 3 specialist agents:
- **Research Agent** — RAG + web search
- **Math Agent** — calculator only
- **Writer Agent** — composes final responses

> Requires Groq API key from: https://console.groq.com

---
## Step 1: Install & Setup

In [ ]:
!pip install -q sentence-transformers chromadb groq
import json
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb

GROQ_API_KEY = 'your_key_here'
MODEL = 'llama-3.3-70b-versatile'
client = Groq(api_key=GROQ_API_KEY)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'✅ Ready — {MODEL}')

---
## Step 2: Build the Knowledge Base

In [ ]:
DOCS = [
    {'id':'d1','source':'refund.txt','text':'Customers may return products within 30 days. Items must be in original condition. Refunds are processed in 5-7 business days.'},
    {'id':'d2','source':'refund.txt','text':'To initiate a refund contact support@company.com. Refund goes to original payment method. Digital products cannot be returned.'},
    {'id':'d3','source':'shipping.txt','text':'Standard shipping takes 3-5 business days. Express shipping takes 1-2 days. Free shipping on orders above $50.'},
    {'id':'d4','source':'pricing.txt','text':'All prices include taxes. Price match guarantee available. Contact sales@company.com with competitor price proof.'},
]

chroma = chromadb.Client()
col = chroma.create_collection('docs', metadata={'hnsw:space':'cosine'})
col.add(
    ids=[d['id'] for d in DOCS],
    embeddings=embedder.encode([d['text'] for d in DOCS]).tolist(),
    documents=[d['text'] for d in DOCS],
    metadatas=[{'source':d['source']} for d in DOCS]
)
print(f'✅ {col.count()} chunks stored')

---
## Step 3: Define Tool Functions

In [ ]:
def search_knowledge_base(query: str) -> str:
    qvec = embedder.encode(query).tolist()
    res = col.query(query_embeddings=[qvec], n_results=2, include=['documents'])
    return '\n'.join(res['documents'][0])

def search_web(query: str) -> str:
    return f'[Simulated web result for: {query}] Python is a popular programming language.'

def calculate(a: float, b: float, operation: str) -> str:
    ops = {'add':a+b,'subtract':a-b,'multiply':a*b,'divide':a/b if b else 'div/0'}
    return str(ops.get(operation, 'unknown op'))

TOOL_REGISTRY = {'search_knowledge_base':search_knowledge_base,'search_web':search_web,'calculate':calculate}

def execute_tool(name, args):
    try:
        return str(TOOL_REGISTRY[name](**args))
    except Exception as e:
        return f'Error: {e}'

print('✅ Tools ready')

---
## Step 4: Build Each Specialist Agent

Each agent is a function that runs its own LLM call with its own system prompt and tools.

In [ ]:
RESEARCH_TOOLS = [
    {'type':'function','function':{'name':'search_knowledge_base','description':'Search company documents.',
     'parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}}},
    {'type':'function','function':{'name':'search_web','description':'Search the web for general facts.',
     'parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}}},
]

MATH_TOOLS = [
    {'type':'function','function':{'name':'calculate','description':'Perform arithmetic: add, subtract, multiply, divide.',
     'parameters':{'type':'object','properties':{
       'a':{'type':'number'},'b':{'type':'number'},
       'operation':{'type':'string','description':'add|subtract|multiply|divide'}
     },'required':['a','b','operation']}}},
]

def run_worker(system_prompt: str, task: str, tools: list = None) -> str:
    """Run a single specialist agent with its own system prompt and tools."""
    messages = [
        {'role':'system','content':system_prompt},
        {'role':'user','content':task}
    ]
    kwargs = {'model':MODEL,'messages':messages,'temperature':0}
    if tools:
        kwargs['tools'] = tools
        kwargs['tool_choice'] = 'auto'

    for _ in range(5):
        resp = client.chat.completions.create(**kwargs)
        msg = resp.choices[0].message
        if not (hasattr(msg,'tool_calls') and msg.tool_calls):
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)
            messages.append({'role':'tool','tool_call_id':tc.id,'content':result})
    return 'Max steps reached'

def research_agent(task: str) -> str:
    print(f'  [Research Agent] Task: {task[:60]}')
    result = run_worker(
        'You are a research specialist. Use search_knowledge_base for company questions '
        'and search_web for general questions. Return key facts only.',
        task, tools=RESEARCH_TOOLS
    )
    print(f'  [Research Agent] Done.')
    return result

def math_agent(task: str) -> str:
    print(f'  [Math Agent] Task: {task[:60]}')
    result = run_worker(
        'You are a math specialist. Always use the calculate tool for arithmetic. '
        'Return only the formula and result.',
        task, tools=MATH_TOOLS
    )
    print(f'  [Math Agent] Done.')
    return result

def writer_agent(task: str) -> str:
    print(f'  [Writer Agent] Composing...')
    result = run_worker(
        'You are a professional writer. Write clear, helpful, and concise responses based on the given information. '
        'Do not add information not provided to you.',
        task
    )
    print(f'  [Writer Agent] Done.')
    return result

print('✅ 3 specialist agents ready')

---
## Step 5: Build the Orchestrator

The orchestrator reads the user request, breaks it into subtasks, calls the right agents, then combines results.

In [ ]:
ORCHESTRATOR_SYSTEM = """You are an orchestrator agent. Your job is to:
1. Analyze the user's request
2. Output a plan as JSON with this structure:
   {"tasks": [{"agent": "research|math|writer", "task": "specific task description"}]}
3. Each subtask must be self-contained and clearly describe what to do.

Available agents:
- research: finds information from company documents or web
- math: performs calculations
- writer: writes the final response when given all needed information

Rules:
- Always end with a writer task that receives all previous results.
- If only a simple answer is needed, use just one writer task.
- Output ONLY valid JSON, no explanation."""

def orchestrator(user_request: str) -> str:
    """Main orchestrator: plan → delegate → combine."""
    print(f'\n{"="*60}')
    print(f'USER: {user_request}')
    print(f'{"="*60}')

    # Step 1: Plan
    print('\n[Orchestrator] Planning...')
    plan_resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':'system','content':ORCHESTRATOR_SYSTEM},
            {'role':'user','content':user_request}
        ],
        temperature=0
    )
    plan_text = plan_resp.choices[0].message.content.strip()

    # Parse JSON plan
    try:
        plan = json.loads(plan_text)
    except:
        # Try to extract JSON if wrapped in markdown
        import re
        match = re.search(r'\{.*\}', plan_text, re.DOTALL)
        plan = json.loads(match.group()) if match else {'tasks':[{'agent':'writer','task':user_request}]}

    print(f'[Orchestrator] Plan: {len(plan["tasks"])} tasks')
    for t in plan['tasks']:
        print(f'  → [{t["agent"]}] {t["task"][:60]}')

    # Step 2: Execute tasks in order, collecting results
    results = []
    AGENT_MAP = {'research':research_agent,'math':math_agent,'writer':writer_agent}

    for i, task_item in enumerate(plan['tasks']):
        agent_name = task_item['agent']
        task_desc  = task_item['task']

        # For the final writer task, inject all previous results
        if agent_name == 'writer' and results:
            context = '\n\n'.join([f'--- Result {j+1} ---\n{r}' for j, r in enumerate(results)])
            task_desc = f'{task_desc}\n\nUse this information:\n{context}'

        agent_fn = AGENT_MAP.get(agent_name, writer_agent)
        result = agent_fn(task_desc)
        results.append(result)

    final = results[-1]
    print(f'\nAGENT: {final}')
    return final

print('✅ Orchestrator ready')

---
## Step 6: Test — Single Agent Routing

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️ Paste your Groq API key in Step 1')
else:
    # Should route to Research Agent only
    orchestrator('What is our refund policy?')

---
## Step 7: Test — Multi-Agent Routing

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️ Paste your Groq API key in Step 1')
else:
    # Should route: Research + Math + Writer
    orchestrator(
        'What is our refund policy? Also calculate the total for '
        '3 orders of $89.50 each. Write a friendly customer email '
        'explaining both.'
    )

---
## Step 8: Test — Shipping + Math

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️ Paste your Groq API key in Step 1')
else:
    # Research (shipping) + Math (cost) + Writer (summary)
    orchestrator(
        'What is our shipping policy? If I have 5 items at $24.99 each, '
        'do I qualify for free shipping? Summarize for me.'
    )

---
## ✅ What You Built

```
User Request
     ↓
Orchestrator makes a plan (JSON)
     ↓
Research Agent  →  finds facts (RAG + web)
Math Agent      →  does calculations
Writer Agent    →  combines into final answer
     ↓
User gets a complete, accurate response
```

| Concept | What We Used |
|---|---|
| Orchestrator | LLM that outputs a JSON plan |
| Specialist agents | Separate LLM calls with focused prompts |
| Tool isolation | Each agent only has relevant tools |
| Result passing | Worker outputs injected into Writer context |

**Next → Module 7: Production Agent** (deploy, monitor, rate limits)